# 02 — Blocking analysis
Reads the blocking cache written by the pipeline (`cache/block_train_*.npz`), so run
`python src/pipeline.py run --data <DATA> --no-test` first. Blocking recall is the ceiling for
everything downstream; this shows recall per view / depth / country and the pairs it misses.

In [ ]:
# --- setup: run from the repo root or from notebooks/ ---
import sys, os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
# pick the first data dir that exists; full data is ~12M rows/split, so EDA defaults to samples
for cand in ["dataset_sample", "dataset_region", "dataset", "fake_dataset"]:
    if (ROOT / cand / "train").exists():
        DATA = ROOT / cand
        break
CACHE = ROOT / "cache"                  # pipeline cache (blocking .npz, oof_train.pkl)
CACHE = ROOT / "cache"
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
print("data:", DATA)

In [ ]:
import glob
from data import load_prepared, load_ground_truth
from pipeline import owner_array
files = sorted(glob.glob(str(CACHE / "block_train_*.npz")))
print(files)
z = np.load(files[-1]); B = {k: z[k] for k in z.files}
s1, s23 = load_prepared(DATA, "train", str(CACHE))
truth = load_ground_truth(DATA)
present = set(s23.entity_id); ids1 = set(s1.entity_id)
truth = {k: v & present for k, v in truth.items() if k in ids1}
own = owner_array(s1, s23, truth)
y = own[B["i2"]] == B["i1"]
n_true = int((own >= 0).sum())
print(f"pairs={len(y)}  per S23={len(y) / len(s23):.2f}  recall={y.sum() / n_true:.4f}")

## Recall by depth: rank of the true S1 inside each S2/S3 record's candidate list, per view

In [ ]:
from features import group_rank_desc
i2 = B["i2"]
ks = [1, 2, 3, 5, 10, 20]
for v in ["nm", "ad", "full"]:
    r = group_rank_desc(i2, B[f"s_{v}"])[y]
    plt.plot(ks, [(r <= k).sum() / n_true for k in ks], marker="o", label=v)
combo = B["s_nm"] + B["s_ad"] + B["s_full"]
r = group_rank_desc(i2, combo)[y]
plt.plot(ks, [(r <= k).sum() / n_true for k in ks], marker="o", label="nm+ad+full")
plt.xlabel("rank of true S1 among the S23's candidates"); plt.ylabel("recall"); plt.grid(alpha=.3); plt.legend(); plt.show()

In [ ]:
# recall by country and source
linked = np.flatnonzero(own >= 0)
found = np.zeros(len(s23), bool); found[B["i2"][y]] = True
d = pd.DataFrame({"country": s23.country_n.values[linked], "src": s23.src.values[linked], "found": found[linked]})
display(d.groupby("country")["found"].agg(["mean", "size"]))
display(d.groupby("src")["found"].agg(["mean", "size"]))

## True pairs that blocking misses — read these, then fix normalisation or add a view

In [ ]:
miss = linked[~found[linked]]
rows = [dict(s1_name=s1.name_core.values[own[j]], s23_name=s23.name_core.values[j],
             s1_addr=s1.addr_n.values[own[j]], s23_addr=s23.addr_n.values[j]) for j in miss[:40]]
print(len(miss), "missed links"); pd.DataFrame(rows)

In [ ]:
# candidates per S23 / per S1
pd.Series(np.bincount(B["i2"], minlength=len(s23))).describe()